# Paper demo: Table 8 Qwen intervention

This notebook runs the paper's **Qwen 3.5 9B Base, layer 26, component C9** English/French case study. It reproduces both directions of the intervention: steering an English continuation toward French and ablating C9 from a French prompt.

## Runtime requirement

Use an **A100-class or other high-memory CUDA GPU**. A free Colab T4 does not have enough VRAM for the unquantized paper model. You can also download this notebook and run it locally on a sufficiently capable GPU.

In [ ]:
%pip install -q icalens==0.4.0

In [ ]:
import torch
from icalens import ICALens

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required. Select an A100 runtime or run locally.")
total_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"GPU: {torch.cuda.get_device_name(0)} ({total_gb:.1f} GiB)")
if total_gb < 24:
    raise RuntimeError("This paper-exact Qwen 9B demo needs a high-memory GPU (A100 recommended); a T4 is insufficient.")

lens = ICALens.from_pretrained("sida/icalens-qwen3.5-9b-base-pile10k")
layer, component = 26, 9
display(lens.component_profile(layer=layer, component=component))

In [ ]:
english_prompt = "Deep learning is a machine learning technique that"
baseline_en = lens.generate(english_prompt, document_framing="model", max_new_tokens=24)
steered_fr = lens.generate(
    english_prompt, layer=layer,
    initial_steer=(component, -40.0, -40.0), steer=(component, -20.0, -20.0),
    steering_scope="current-position", document_framing="model", max_new_tokens=24,
)
print("English baseline:", english_prompt + baseline_en)
print("English → French:", english_prompt + steered_fr)

In [ ]:
french_prompt = "L’apprentissage profond est une technique d’apprentissage automatique qui"
baseline_fr = lens.generate(french_prompt, document_framing="model", max_new_tokens=24)
ablated_en = lens.generate(
    french_prompt, layer=layer, clamp=(component, 0.0),
    document_framing="model", max_new_tokens=24,
)
print("French baseline:", french_prompt + baseline_fr)
print("French → English after C9 ablation:", french_prompt + ablated_en)

The component is not merely correlated with a language label: adding its writing direction can move an English prompt into French, while clamping the same coordinate can reverse a French continuation toward English. Exact decoding can vary with package and Transformers versions.